# 🚢 Container OCR: Precision Mode Demo

This notebook demonstrates the **Champion Model** for maritime container recognition. It uses a custom YOLOv8 detector combined with **EasyOCR** and our **ISO 6346 Smart Correction** algorithm to achieve maximum accuracy.

In [ ]:
import os
import cv2
import torch
import matplotlib.pyplot as plt
from pathlib import Path
import sys

# Ensure project root is in path
sys.path.append(os.getcwd())

from src.inference.pipeline import load_models, read_easyocr
from src.utils.formatters import format_container_code
from src.config import CONTAINER_TEST_DATA

## 1. Load the Models
We load the YOLOv8 block detector and initialize the EasyOCR engine.

In [ ]:
os.environ['DETECT_MODE'] = 'container'
os.environ['OCR_TYPE'] = 'precise'
device = "cuda" if torch.cuda.is_available() else "cpu"

yolo_model, ocr_engine, _ = load_models(device, mode='container')
print(f"✓ Models loaded on {device}")

## 2. Test an Image
Pick an image from the test set or provide your own path.

In [ ]:
# Select a random image from the test set
test_images = list(CONTAINER_TEST_DATA.glob('*.jpg'))
if not test_images:
    print("No test images found in the config path.")
else:
    img_path = str(test_images[0])
    img = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Stage 1: Detection
    results = yolo_model(img, verbose=False)
    
    plt.figure(figsize=(12, 8))
    plt.imshow(img_rgb)
    plt.title("Original Image")
    plt.axis('off')
    plt.show()
    
    for result in results:
        for box in result.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
            pad = 15
            crop = img[max(0, y1-pad):min(img.shape[0], y2+pad), max(0, x1-pad):min(img.shape[1], x2+pad)]
            
            # Stage 2: OCR + Smart Correction
            raw_text = read_easyocr(crop, ocr_engine)
            corrected_text = format_container_code(raw_text)
            
            print(f"--- Result ---")
            print(f"Raw OCR Output:    {raw_text}")
            print(f"Smart Corrected:   {corrected_text}")